# Paired bootstrap significance testing for the forgetting-mitigation comparison

Compares the SQuAD-only baseline, sequential fine-tune, joint training, and distillation-regularized fine-tune against each other, using paired bootstrap resampling (Berg-Kirkpatrick et al., 2012) over per-question EM/F1.

Reuses `get_raw_scores()` from the official `evaluate-v2.0.py` so per-question scores match exactly what the course scorer reports in aggregate. The baseline's SQuAD dev predictions were never saved to Drive (see the gotcha in `bert_baseline_results.md`), so this notebook regenerates them from the `qa_model` checkpoint if missing, so the baseline can be included in the significance tests, not just the other three models against each other.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import importlib.util
import json
import os

import numpy as np

PROJECT_DIR = "/content/drive/MyDrive/CS4248 Group Project"
DATA_DIR = os.path.join(PROJECT_DIR, "qa")

BASE_MODEL_DIR = os.path.join(PROJECT_DIR, "qa_model")  # SQuAD-only baseline checkpoint

SQUAD_DEV_FILE = os.path.join(DATA_DIR, "dev-v1.1.json")
NEWSQA_DEV_FILE = os.path.join(DATA_DIR, "dev-newsqa.json")
EVAL_SCRIPT = os.path.join(DATA_DIR, "evaluate-v2.0.py")

MAX_LENGTH = 384
DOC_STRIDE = 128
BATCH_SIZE = 8
N_BEST = 20
MAX_ANSWER_LENGTH = 30

PREDICTIONS_SQUAD_BASELINE_FILE = os.path.join(DATA_DIR, "predictions_squad_baseline.json")

# predictions files, per model, per dev set (fill in any missing ones)
SQUAD_PREDICTIONS = {
    "baseline": PREDICTIONS_SQUAD_BASELINE_FILE,
    "sequential": os.path.join(DATA_DIR, "predictions_squad_after_newsqa_finetune.json"),
    "joint": os.path.join(DATA_DIR, "predictions_squad_after_joint.json"),
    "distillation": os.path.join(DATA_DIR, "predictions_squad_after_newsqa_distill.json"),
}

NEWSQA_PREDICTIONS = {
    "sequential": os.path.join(DATA_DIR, "predictions_newsqa_finetuned.json"),
    "joint": os.path.join(DATA_DIR, "predictions_newsqa_after_joint.json"),
    "distillation": os.path.join(DATA_DIR, "predictions_newsqa_distill.json"),
}

N_RESAMPLES = 10000
SEED = 42

## Regenerate the missing baseline SQuAD predictions

Skips entirely if `predictions_squad_baseline.json` already exists on Drive. Otherwise loads the SQuAD-only `qa_model` checkpoint and runs the same inference and postprocessing logic used in the other notebooks, so the result is directly comparable.

In [3]:
if not os.path.exists(PREDICTIONS_SQUAD_BASELINE_FILE):
    import collections
    import subprocess

    import torch
    from datasets import Dataset
    from transformers import AutoTokenizer, AutoModelForQuestionAnswering

    def load_squad_json(path):
        with open(path) as f:
            squad = json.load(f)
        ids, questions, contexts, answers = [], [], [], []
        for article in squad["data"]:
            for para in article["paragraphs"]:
                context = para["context"]
                for qa in para["qas"]:
                    ids.append(qa["id"])
                    questions.append(qa["question"])
                    contexts.append(context)
                    answers.append(
                        {
                            "text": [a["text"] for a in qa["answers"]],
                            "answer_start": [a["answer_start"] for a in qa["answers"]],
                        }
                    )
        return Dataset.from_dict(
            {"id": ids, "question": questions, "context": contexts, "answers": answers}
        )

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR)
    model = AutoModelForQuestionAnswering.from_pretrained(BASE_MODEL_DIR).to(device)
    model.eval()

    squad_dev_raw = load_squad_json(SQUAD_DEV_FILE)

    def prepare_validation_features(examples):
        questions = [q.lstrip() for q in examples["question"]]
        tokenized = tokenizer(
            questions,
            examples["context"],
            max_length=MAX_LENGTH,
            truncation="only_second",
            stride=DOC_STRIDE,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding="max_length",
        )
        sample_map = tokenized.pop("overflow_to_sample_mapping")
        example_ids = []
        for i in range(len(tokenized["input_ids"])):
            sample_idx = sample_map[i]
            example_ids.append(examples["id"][sample_idx])
            sequence_ids = tokenized.sequence_ids(i)
            tokenized["offset_mapping"][i] = [
                o if sequence_ids[k] == 1 else None
                for k, o in enumerate(tokenized["offset_mapping"][i])
            ]
        tokenized["example_id"] = example_ids
        return tokenized

    squad_validation_dataset = squad_dev_raw.map(
        prepare_validation_features,
        batched=True,
        remove_columns=squad_dev_raw.column_names,
    )

    def run_inference(model, dataset, batch_size=BATCH_SIZE):
        eval_set_for_model = dataset.remove_columns(["example_id", "offset_mapping"])
        eval_set_for_model.set_format("torch")
        all_start_logits, all_end_logits = [], []
        with torch.no_grad():
            for i in range(0, len(eval_set_for_model), batch_size):
                batch = eval_set_for_model[i : i + batch_size]
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**batch)
                all_start_logits.append(outputs.start_logits.cpu().numpy())
                all_end_logits.append(outputs.end_logits.cpu().numpy())
        return np.concatenate(all_start_logits, axis=0), np.concatenate(all_end_logits, axis=0)

    def postprocess_qa_predictions(examples, features, all_start_logits, all_end_logits):
        example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
        features_per_example = collections.defaultdict(list)
        for i, feature_id in enumerate(features["example_id"]):
            features_per_example[example_id_to_index[feature_id]].append(i)

        predictions = {}
        for example_index, example in enumerate(examples):
            context = example["context"]
            feature_indices = features_per_example[example_index]
            valid_answers = []
            for feature_index in feature_indices:
                start_logit = all_start_logits[feature_index]
                end_logit = all_end_logits[feature_index]
                offsets = features[feature_index]["offset_mapping"]
                start_indexes = np.argsort(start_logit)[-1 : -N_BEST - 1 : -1].tolist()
                end_indexes = np.argsort(end_logit)[-1 : -N_BEST - 1 : -1].tolist()
                for start_index in start_indexes:
                    for end_index in end_indexes:
                        if (
                            start_index >= len(offsets)
                            or end_index >= len(offsets)
                            or offsets[start_index] is None
                            or offsets[end_index] is None
                        ):
                            continue
                        if end_index < start_index or end_index - start_index + 1 > MAX_ANSWER_LENGTH:
                            continue
                        start_char, end_char = offsets[start_index][0], offsets[end_index][1]
                        valid_answers.append(
                            {
                                "score": start_logit[start_index] + end_logit[end_index],
                                "text": context[start_char:end_char],
                            }
                        )
            if valid_answers:
                best = max(valid_answers, key=lambda x: x["score"])
                predictions[example["id"]] = best["text"]
            else:
                predictions[example["id"]] = ""
        return predictions

    print("Running inference on SQuAD dev with the baseline qa_model checkpoint...")
    start_logits, end_logits = run_inference(model, squad_validation_dataset)
    baseline_predictions = postprocess_qa_predictions(
        squad_dev_raw, squad_validation_dataset, start_logits, end_logits
    )
    with open(PREDICTIONS_SQUAD_BASELINE_FILE, "w") as f:
        json.dump(baseline_predictions, f)

    print("Sanity check against the known baseline (80.92 EM / 88.40 F1):")
    result = subprocess.run(
        ["python", EVAL_SCRIPT, SQUAD_DEV_FILE, PREDICTIONS_SQUAD_BASELINE_FILE],
        check=True, capture_output=True, text=True,
    )
    print(result.stdout)
else:
    print(f"{PREDICTIONS_SQUAD_BASELINE_FILE} already exists, skipping inference.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

Running inference on SQuAD dev with the baseline qa_model checkpoint...
Sanity check against the known baseline (80.92 EM / 88.40 F1):
{
  "exact": 81.07852412488174,
  "f1": 88.57866789658426,
  "total": 10570,
  "HasAns_exact": 81.07852412488174,
  "HasAns_f1": 88.57866789658426,
  "HasAns_total": 10570
}



In [4]:
# load get_raw_scores from the official scorer without renaming the file
spec = importlib.util.spec_from_file_location("squad_eval", EVAL_SCRIPT)
squad_eval = importlib.util.module_from_spec(spec)
spec.loader.exec_module(squad_eval)


def per_question_scores(dev_file, predictions_file):
    with open(dev_file) as f:
        dataset = json.load(f)["data"]
    with open(predictions_file) as f:
        preds = json.load(f)
    exact_raw, f1_raw = squad_eval.get_raw_scores(dataset, preds)
    return exact_raw, f1_raw

In [5]:
def paired_bootstrap(scores_a, scores_b, qids, n_resamples=N_RESAMPLES, seed=SEED):
    """
    Berg-Kirkpatrick et al. (2012)-style paired bootstrap.
    scores_a, scores_b: dict qid -> score (same metric, same dev set).
    Returns (mean_a, mean_b, observed_diff, p_value, ci_low, ci_high) where
    p_value is the two-sided probability that the sign of the difference
    flips under resampling, and the CI is a 95% percentile interval on the
    resampled a-minus-b difference.
    """
    rng = np.random.default_rng(seed)
    a = np.array([scores_a[q] for q in qids])
    b = np.array([scores_b[q] for q in qids])
    n = len(qids)

    observed_diff = a.mean() - b.mean()

    diffs = np.empty(n_resamples)
    for i in range(n_resamples):
        idx = rng.integers(0, n, size=n)
        diffs[i] = a[idx].mean() - b[idx].mean()

    if observed_diff >= 0:
        p_value = 2 * np.mean(diffs <= 0)
    else:
        p_value = 2 * np.mean(diffs >= 0)
    p_value = min(p_value, 1.0)

    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
    return a.mean(), b.mean(), observed_diff, p_value, ci_low, ci_high

In [6]:
def compare_all_pairs(dev_file, predictions_by_model, dataset_name):
    exact_by_model, f1_by_model = {}, {}
    for model_name, pred_file in predictions_by_model.items():
        exact_by_model[model_name], f1_by_model[model_name] = per_question_scores(dev_file, pred_file)

    qids = sorted(set.intersection(*(set(d.keys()) for d in exact_by_model.values())))
    print(f"\n=== {dataset_name} ({len(qids)} questions) ===")

    model_names = list(predictions_by_model.keys())
    for metric_name, scores_by_model in (("EM", exact_by_model), ("F1", f1_by_model)):
        for i in range(len(model_names)):
            for j in range(i + 1, len(model_names)):
                m_a, m_b = model_names[i], model_names[j]
                mean_a, mean_b, diff, p, lo, hi = paired_bootstrap(
                    scores_by_model[m_a], scores_by_model[m_b], qids
                )
                sig = "significant" if p < 0.05 else "not significant"
                print(
                    f"{metric_name}: {m_a} ({mean_a*100:.2f}) vs {m_b} ({mean_b*100:.2f}) "
                    f"diff={diff*100:+.2f} 95% CI=[{lo*100:+.2f}, {hi*100:+.2f}] "
                    f"p={p:.4f} ({sig} at alpha=0.05)"
                )

In [7]:
compare_all_pairs(SQUAD_DEV_FILE, SQUAD_PREDICTIONS, "SQuAD dev (retention)")
compare_all_pairs(NEWSQA_DEV_FILE, NEWSQA_PREDICTIONS, "NewsQA dev (adaptation)")


=== SQuAD dev (retention) (10570 questions) ===
EM: baseline (81.08) vs sequential (74.84) diff=+6.23 95% CI=[+5.51, +6.95] p=0.0000 (significant at alpha=0.05)
EM: baseline (81.08) vs joint (81.12) diff=-0.04 95% CI=[-0.59, +0.52] p=0.9242 (not significant at alpha=0.05)
EM: baseline (81.08) vs distillation (80.08) diff=+1.00 95% CI=[+0.51, +1.49] p=0.0004 (significant at alpha=0.05)
EM: sequential (74.84) vs joint (81.12) diff=-6.27 95% CI=[-7.01, -5.54] p=0.0000 (significant at alpha=0.05)
EM: sequential (74.84) vs distillation (80.08) diff=-5.23 95% CI=[-5.84, -4.62] p=0.0000 (significant at alpha=0.05)
EM: joint (81.12) vs distillation (80.08) diff=+1.04 95% CI=[+0.44, +1.64] p=0.0012 (significant at alpha=0.05)
F1: baseline (88.58) vs sequential (85.44) diff=+3.14 95% CI=[+2.67, +3.60] p=0.0000 (significant at alpha=0.05)
F1: baseline (88.58) vs joint (88.58) diff=+0.00 95% CI=[-0.37, +0.38] p=1.0000 (not significant at alpha=0.05)
F1: baseline (88.58) vs distillation (88.06) di